# preparation et exploration des donnees

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

In [ ]:
import pandas as pd

commandes = pd.read_excel('data.xlsx')

commandes

## identifier les principales caracteristiques des donnees

In [ ]:
commandes.info()
commandes.describe()
commandes.shape
commandes.ndim

In [ ]:
commandes.describe()

In [ ]:
# # commandes.isna().sum()
commandes.duplicated().sum()
# (commandes['Quantity'] <= 0).sum()
# (commandes['UnitPrice'] <= 0).sum()

## realiser le nettoyage necessaire 

In [ ]:
commandes = commandes.dropna(subset=['CustomerID'])
commandes = commandes[~commandes['InvoiceNo'].astype(str).str.startswith("C")]
commandes['CustomerID'] = commandes['CustomerID'].astype(int)
commandes = commandes.drop(columns=['Description', 'StockCode', 'Country'])
commandes['InvoiceDate'] = pd.to_datetime(commandes['InvoiceDate'])
commandes = commandes[commandes['Quantity'] > 0]
commandes = commandes[commandes['UnitPrice'] > 0]
commandes['TotalPrice'] = commandes['Quantity'] * commandes['UnitPrice']

# construction des indicateurs RFM

In [ ]:
date_reference = commandes['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = commandes.groupby('CustomerID').agg(
    Recence=('InvoiceDate', lambda x: (date_reference - x.max()).days),
    Frequence=('InvoiceNo', 'nunique'),
    Montant=('TotalPrice', 'sum')
).reset_index()



In [ ]:
# -------------------------
# 1. Create transformed RFM
# -------------------------

rfm_transformed = rfm.copy()

# Recence: make recent customers have larger values
rfm_transformed['Recence'] = -rfm_transformed['Recence']

# Log-transform skewed variables
rfm_transformed['Frequence'] = np.log1p(
    rfm_transformed['Frequence']
)

rfm_transformed['Montant'] = np.log1p(
    rfm_transformed['Montant']
)

In [ ]:
plt.hist(rfm_transformed['Montant'],bins=30)
plt.show()

# preparation des variables pour le clustring

In [ ]:


features = ['Recence', 'Frequence', 'Montant']

scaler = StandardScaler()

rfm_scaled_values = scaler.fit_transform(
    rfm_transformed[features]
)

rfm_scaled = pd.DataFrame(
    rfm_scaled_values,
    columns=features,
    index=rfm.index
)

rfm_scaled['Cluster'] = rfm['Cluster']

# segmentation avec KMeans

In [ ]:
inertias = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)

In [ ]:
plt.plot(range(2, 11), inertias, marker='o')

plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Inertie')
plt.title('Méthode du coude')

plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

rfm_scaled['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(rfm_scaled)

    score = silhouette_score(
        rfm_scaled,
        labels
    )

    silhouette_scores.append(score)

    print(f"k={k} : silhouette={score:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker='o'
)

plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Silhouette Coefficient')
plt.title('Silhouette Coefficient selon le nombre de clusters')
plt.show()

In [ ]:
rfm.groupby('Cluster').count()['CustomerID']

# profiling et interpretation metier

In [ ]:
ca_total = rfm['Montant'].sum()

In [ ]:
cluster_summary = rfm.groupby('Cluster').agg(
    NombreClients=('CustomerID', 'count'),
    RecenceMoyenne=('Recence', 'mean'),
    FrequenceMoyenne=('Frequence', 'mean'),
    MontantMoyen=('Montant', 'mean'),
    CA=('Montant', 'sum')
).reset_index()

In [ ]:
cluster_summary['ContributionCA'] = (
    cluster_summary['CA'] / ca_total * 100
)

In [ ]:
cluster_summary

cluster 0 --> recence basse + frequence basse + contribution au ca basse
cluster 1 --> recence moyenne + frequence basse + contribution au ca moyenne
cluster 2 --> recence elevee + frequence haute + contribution au ca elevee
